In [ ]:
# Mega Harness + Qwen 3.8 27B Abliterated — Kaggle, uma célula
# Ativa Internet e GPU no Kaggle antes de executar.
# A célula imprime apenas o Base URL e continua ativa para manter o túnel.

import os, re, sys, time, signal, pathlib, shutil, subprocess, urllib.request

MODEL = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:UD-Q4_K_XL"
PORT = 8000
LOCAL = f"http://127.0.0.1:{PORT}"
HOME = os.path.expanduser("~")
LLAMA = os.path.join(HOME, ".local", "bin", "llama")
if not os.path.isfile(LLAMA):
    LLAMA = os.path.join(HOME, ".llama-app", "llama")

# Parar tentativas antigas na mesma sessão Kaggle (por exemplo, célula interrompida).
for proc in pathlib.Path("/proc").iterdir():
    if not proc.name.isdigit():
        continue
    try:
        args = (proc / "cmdline").read_bytes()
        if b"llama" in args and b"serve" in args and str(PORT).encode() in args:
            os.kill(int(proc.name), signal.SIGTERM)
        elif b"cloudflared" in args and b"tunnel" in args:
            os.kill(int(proc.name), signal.SIGTERM)
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        pass
time.sleep(2)

# Instalar o binário CUDA pré-compilado só se ainda não estiver instalado.
if not os.path.isfile(LLAMA):
    subprocess.run(
        "curl -LsSf https://llama.app/install.sh | sh",
        shell=True, executable="/bin/bash", check=True,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    LLAMA = os.path.join(HOME, ".local", "bin", "llama")
    if not os.path.isfile(LLAMA):
        LLAMA = os.path.join(HOME, ".llama-app", "llama")
if not os.path.isfile(LLAMA):
    raise RuntimeError("Não foi possível instalar llama.cpp no Kaggle.")

# -hf reutiliza a cache se já existir; se faltar, descarrega o modelo automaticamente.
# Sem --api-key: permite configurar o Mega Harness só com o Base URL.
server_log = open("/kaggle/working/qwen-server.log", "w")
server = subprocess.Popen(
    [LLAMA, "serve", "-hf", MODEL,
     "--host", "0.0.0.0", "--port", str(PORT),
     "--alias", "qwen3.8-27b-abliterated",
     "--ctx-size", "4096", "--gpu-layers", "all",
     "--tensor-split", "1,1", "--jinja", "--parallel", "1", "--no-webui"],
    stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True
)

# Esperar pelo download e carregamento do modelo.
import urllib.error
ready = False
for _ in range(1800):  # até 30 minutos
    if server.poll() is not None:
        server_log.flush()
        raise RuntimeError("llama-server terminou. Consulta /kaggle/working/qwen-server.log")
    try:
        with urllib.request.urlopen(LOCAL + "/health", timeout=3) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)
if not ready:
    raise TimeoutError("O modelo ainda não ficou pronto. Consulta /kaggle/working/qwen-server.log")

# Instalar cloudflared apenas se faltar.
if not shutil.which("cloudflared"):
    deb = "/kaggle/working/cloudflared.deb"
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        deb
    )
    subprocess.run(["dpkg", "-i", deb], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Criar o túnel HTTPS temporário.
tunnel_log = open("/kaggle/working/qwen-tunnel.log", "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", LOCAL, "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True
)
public_url = None
for _ in range(120):
    tunnel_log.flush()
    log = pathlib.Path("/kaggle/working/qwen-tunnel.log").read_text(errors="ignore")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if match:
        public_url = match.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError("O túnel parou. Consulta /kaggle/working/qwen-tunnel.log")
    time.sleep(1)
if not public_url:
    raise TimeoutError("O túnel não iniciou. Consulta /kaggle/working/qwen-tunnel.log")

# Única saída visível: cola este endereço no campo URL base do Mega Harness.
print(public_url + "/v1", flush=True)

# Manter o runtime e o túnel ativos.
while True:
    time.sleep(60)
